In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth
from collections import Counter
import json
import os
from prefixspan import PrefixSpan
from mlxtend.frequent_patterns import apriori
from pprint import pprint
from sklearn.mixture import GaussianMixture

In [2]:
parent_dir = os.path.dirname(os.getcwd())
SILVER_PATH = parent_dir + "/data/2_silver/" 

In [3]:
def evaluate_threshold(df, stops_df, threshold_seconds, min_support=0.5):
    """
    Hàm này chạy toàn bộ pipeline (cắt chuyến + tìm tuyến) cho một ngưỡng thời gian cụ thể
    và trả về các chỉ số đánh giá (metrics).
    """
    # 1. CẮT CHUYẾN (Theo phiên bản tối ưu Vectorized)
    df = df.sort_values(by=['vehicle', 'datetime'])
    
    # Tính khoảng cách thời gian với dòng ngay trước đó (theo từng xe)
    time_diff_raw = df.groupby('vehicle')['datetime'].diff()
    if pd.api.types.is_timedelta64_dtype(time_diff_raw):
        df['time_diff'] = time_diff_raw.dt.total_seconds()
    else:
        df['time_diff'] = time_diff_raw
    
    is_new_by_gap = (df['time_diff'] > threshold_seconds) | (df['time_diff'].isna())
    
    # 2. CẮT THEO BẾN CUỐI (Terminal State)
    df['is_resting'] = (df['is_terminal'] == True) & (df['station_distance'] <= 50)
    df['prev_is_resting'] = df.groupby('vehicle')['is_resting'].shift(1).fillna(False)
    is_new_by_terminal = (df['prev_is_resting'] == True) & (df['is_resting'] == False)
    
    # ==========================================
    # 3. MỚI: CẮT THEO "NGỦ ĐÔNG" (BẢO TRÌ/HỎNG HÓC)
    # Cảnh báo: Phức tạp hơn một chút vì ta phải đo THỜI GIAN ĐỨNG IM
    # ==========================================
    # Tạo cờ xem xe có đang đứng im không (< 5 km/h)
    df['is_stationary'] = df['speed'] < 5
    
    # Tính thời gian đứng im tích lũy. Nếu xe nhích lên > 5km/h, bộ đếm reset về 0
    # Đây là một trick dùng Groupby kết hợp Cumsum rất kinh điển
    df['move_trigger'] = (df['is_stationary'] == False).cumsum()
    df['stationary_duration'] = df.groupby(['vehicle', 'move_trigger'])['time_diff'].cumsum().fillna(0)
    
    # Nếu thời gian đứng im vượt quá 2 tiếng (7200 giây)
    df['is_long_idle'] = df['stationary_duration'] > 7200
    df['prev_is_long_idle'] = df.groupby('vehicle')['is_long_idle'].shift(1).fillna(False)
    
    # Kích hoạt cắt: Vừa thoát khỏi trạng thái ngủ đông (Bắt đầu lăn bánh rời xưởng/rời chỗ sửa xe)
    is_new_by_idle = (df['prev_is_long_idle'] == True) & (df['is_long_idle'] == False)
    
    # ==========================================
    # KẾT HỢP CẢ 3 LUẬT LẠI
    # ==========================================
    df['is_new_trip'] = is_new_by_gap | is_new_by_terminal | is_new_by_idle
    df['trip_id'] = df.groupby('vehicle')['is_new_trip'].cumsum()
    
    # Clean up các cột tạm thời
    df.drop(columns=[
        'is_resting', 'prev_is_resting', 'is_stationary', 
        'move_trigger', 'stationary_duration', 'is_long_idle', 'prev_is_long_idle', 'is_new_trip'
    ], inplace=True, errors='ignore')

    mask_same_station = df['current_station'] == df['current_station'].shift()
    mask_same_vehicle = df['vehicle'] == df['vehicle'].shift()
    mask_same_trip = df['trip_id'] == df['trip_id'].shift()
    
    df_cleaned = df[~(mask_same_station & mask_same_vehicle & mask_same_trip)]
    transactions_series = df_cleaned.groupby(['vehicle', 'trip_id'], sort=False)['current_station'].agg(list)
    
    # Gom nhóm theo xe để tìm tuyến
    trips_grouped = transactions_series.reset_index()
    

    print("Đang tạo từ điển tra cứu tuyến đường...")
    route_dict = {}
    # Loại bỏ các trạm không có thông tin tuyến để vòng lặp nhẹ hơn
    valid_stops = stops_df.dropna(subset=['Routes'])

    for _, row in valid_stops.iterrows():
        # Chuyển "152, 156D" thành list ['152', '156D']
        routes_list = [r.strip() for r in str(row['Routes']).split(',')]
        route_dict[row['Name']] = routes_list
        
    # 2. TÌM TUYẾN (Dùng phương pháp đếm Counter siêu tốc)
    from collections import Counter
    results = []

    print("Bắt đầu tìm Tuyến lõi")
    for vehicle, vehicle_trips in trips_grouped.groupby('vehicle'):
        # Danh sách các chuyến đi (transactions là list of lists)
        transactions = vehicle_trips['current_station'].tolist()
        num_trips = len(transactions)
        
        if num_trips < 2:
            continue
            
        # =====================================================================
        # GIẢI PHÁP MỚI: Đếm tần suất trạm đơn lẻ thay vì tìm tổ hợp
        # =====================================================================
        station_counts = Counter()
        
        for trip in transactions:
            # Dùng set(trip) để mỗi chuyến đi chỉ đếm trạm 1 lần (loại bỏ trạm lặp nội bộ)
            station_counts.update(set(trip))
            
        # Số chuyến đi tối thiểu để một trạm được coi là "Trạm Lõi"
        min_required_trips = num_trips * min_support  # Ví dụ: 5 chuyến * 0.5 = 2.5 chuyến
        
        # Lọc ra các trạm đạt chuẩn
        core_stations = [station for station, count in station_counts.items() if count >= min_required_trips]
        
        if not core_stations:
            continue
            
        # =====================================================================
        # Đối chiếu Tập trạm Lõi với Dictionary (Giữ nguyên logic cũ)
        # =====================================================================
        candidate_routes = []
        for station in core_stations:
            routes_list = route_dict.get(station, [])
            candidate_routes.extend(routes_list)
        
        if candidate_routes:
            most_common_route = Counter(candidate_routes).most_common(1)[0][0]
            results.append({
                'vehicle': vehicle, 
                'Inferred_Route': most_common_route,
                'Total_Trips': num_trips,
                'Core_Stations_Count': len(core_stations)
            })
            
    result_df = pd.DataFrame(results)
    
    # 3. TÍNH TOÁN METRICS ĐỂ ĐÁNH GIÁ
    if result_df.empty:
        return {'Threshold_Mins': threshold_seconds/60, 'Assigned_Vehicles': 0, 'Avg_Trips': 0, 'Avg_Core_Stations': 0}
        
    metrics = {
        'Threshold_Mins': threshold_seconds / 60,
        'Assigned_Vehicles': len(result_df), # Quan trọng nhất: Càng nhiều xe được gán tuyến càng tốt
        'Avg_Trips': result_df['Total_Trips'].mean(), # Không nên quá cao (bị băm nát)
        'Avg_Core_Stations': result_df['Core_Stations_Count'].mean() # Quan trọng nhì: Càng dài càng đại diện chuẩn
    }
    return metrics

In [4]:
def GMM_fit(df):
    # Lọc bỏ các khoảng thời gian quá nhỏ (< 60s) vì chắc chắn không phải là cắt chuyến
    # và quá lớn (> 3 tiếng) để thuật toán không bị nhiễu bởi outlier qua đêm
    df = df.sort_values(by=['vehicle', 'datetime']).copy()
    df['time_diff'] = df.groupby('vehicle', sort=False)['datetime'].diff()
    time_diff_series = df['time_diff']
    valid_diffs = time_diff_series[(time_diff_series > 60) & (time_diff_series < 10800)].dropna()
    
    # Chuẩn bị dữ liệu cho Sklearn
    X = valid_diffs.values.reshape(-1, 1)
    
    # Khởi tạo GMM với 2 cụm (Kẹt xe vs Nghỉ tại bến)
    gmm = GaussianMixture(n_components=2, random_state=42)
    gmm.fit(X)
    
    # Lấy giá trị trung bình (mean) của 2 cụm
    means = gmm.means_.flatten()
    means.sort() # Sắp xếp: means[0] là trung bình kẹt xe, means[1] là trung bình nghỉ bến
    
    # Thuật toán đơn giản tìm ranh giới (Decision Boundary): 
    # Có thể lấy điểm giữa của 2 means, hoặc dùng công thức giao điểm Gauss
    # Ở đây lấy trung bình cộng để ước lượng ngưỡng (Threshold)
    optimal_threshold = (means[0] + means[1]) / 2
    
    print(f"GMM đã phân cụm: \n - Trễ nội bộ trung bình: {means[0]/60:.1f} phút \n - Nghỉ bến trung bình: {means[1]/60:.1f} phút")
    print(f"=> Ngưỡng thời gian cắt chuyến tối ưu: {optimal_threshold/60:.1f} phút ({optimal_threshold:.0f} giây)")
    
    return optimal_threshold


In [5]:
from scipy.stats import gaussian_kde
from scipy.signal import argrelextrema
import numpy as np
def KDE_fit(df):
    """
    Dùng KDE để tìm 'đáy thung lũng' giữa 2 đỉnh phân phối thời gian trễ.
    """
    df = df.sort_values(by=['vehicle', 'datetime']).copy()
    df['time_diff'] = df.groupby('vehicle', sort=False)['datetime'].diff()
    time_diff_series = df['time_diff']
    print("1. Đang tiền xử lý và lọc nhiễu dữ liệu...")
    # Lọc bỏ các khoảng nhỏ hơn 1 phút (chạy bình thường) và lớn hơn 3 tiếng (qua đêm)
    # Đồng thời đổi sang đơn vị PHÚT để dễ vẽ biểu đồ và tính toán
    valid_diffs = time_diff_series[(time_diff_series > 60) & (time_diff_series < 10800)].dropna() / 60.0
    
    print("2. Đang xây dựng đường cong KDE (Làm mượt biểu đồ)...")
    # Tạo một trục X giả lập từ giá trị nhỏ nhất đến lớn nhất (chia làm 1000 điểm mượt mà)
    x_grid = np.linspace(valid_diffs.min(), valid_diffs.max(), 1000)
    
    # Huấn luyện mô hình KDE (Kernel Density Estimation)
    # bw_method (Bandwidth): Độ mượt của đường cong. Có thể tinh chỉnh số này (0.1, 0.2, 'scott', 'silverman')
    kde = gaussian_kde(valid_diffs, bw_method=0.15) 
    
    # Tính toán giá trị trục Y (Mật độ) cho 1000 điểm trên trục X
    density = kde.evaluate(x_grid)
    
    print("3. Đang dò tìm Đáy thung lũng (Local Minimum)...")
    # argrelextrema dùng để tìm các cực tiểu địa phương (các đáy)
    minima_indices = argrelextrema(density, np.less)[0]
    
    # Lọc các đáy rơi vào vùng thời gian nghỉ hợp lý của xe buýt (Ví dụ từ 15 phút đến 90 phút)
    # Tránh việc thuật toán bắt nhầm các gợn sóng nhỏ ở phần đầu hoặc phần đuôi biểu đồ
    valid_minima = [x_grid[i] for i in minima_indices if 15 < x_grid[i] < 90]
    
    if not valid_minima:
        print("⚠️ Không tìm thấy thung lũng rõ ràng. Có thể do Bandwidth chưa hợp lý. Dùng mặc định 45 phút.")
        optimal_threshold_mins = 45.0
    else:
        # Nếu có nhiều đáy, ta chọn đáy có giá trị thời gian nhỏ nhất nằm trong vùng hợp lý
        optimal_threshold_mins = valid_minima[0]
        
    optimal_threshold_secs = optimal_threshold_mins * 60
    print(f"HOÀN TẤT! Ngưỡng cắt chuyến tối ưu là: {optimal_threshold_mins:.1f} phút ({optimal_threshold_secs:.0f} giây)")
    return optimal_threshold_secs

In [ ]:
def fine_tune_segmentation(silver_df, stops_df):
    print("Bắt đầu Fine-tuning quá trình phân rã chuyến đi...")
    
    # Other algorithm
    GMM_threshold = GMM_fit(silver_df)
    KDE_threshold = KDE_fit(silver_df)
    # Thử nghiệm với các ngưỡng: 15p, 30p, 45p, 60p, 90p
    # test_thresholds = [GMM_threshold, KDE_threshold, 900, 1800, 2700, 3600, 5400, 7200, 10800] 
    test_thresholds = [GMM_threshold, KDE_threshold]
    test_thresholds = test_thresholds + [x for x in range(900, 7200, 600)]
    evaluation_results = []
    
    for th in test_thresholds:
        print(f"Đang đánh giá ngưỡng {th/60} phút...")
        metrics = evaluate_threshold(silver_df, stops_df, threshold_seconds=th, min_support=0.5)
        evaluation_results.append(metrics)
        
    # Tạo bảng báo cáo
    report_df = pd.DataFrame(evaluation_results)
    print("BẢNG KẾT QUẢ FINE-TUNING:")
    print(report_df.to_string(index=False))
    
    return report_df

In [7]:
def load_data():
    
    with open(SILVER_PATH+"bus_station_data.json", "r", encoding="utf-8") as f:
        station_data = json.load(f)
        
    station_df = pd.DataFrame(station_data)
    gps_df = pd.read_parquet(SILVER_PATH+"bus_gps_data.parquet", engine="pyarrow", )
    return gps_df, station_df

def create_stops_from_silver(df):

    df = df[['Name', 'Routes']].copy()
    list_routes = ["70-5", "61-7", "156D", "156V", "169D", "169V", "163D", "163V", "1", "164D", "164V", "57", "167D", "167V", "152", "27", "148", "55", "151", "30", "122", "3", "72", "45", "88", "32", "91", "93", "24", "50", "90"]
    
    def filter_and_join_routes(routes_str):

        if not isinstance(routes_str, str):
            return ""
        filtered_routes = [route.strip() for route in routes_str.split(',') if route.strip() in list_routes]
        return ','.join(filtered_routes)
    
    df['Routes'] = df['Routes'].apply(filter_and_join_routes)

    return df

In [8]:
silver_df, stops_df = load_data()
stops_df = create_stops_from_silver(stops_df)
report = fine_tune_segmentation(silver_df, stops_df)

Bắt đầu Fine-tuning quá trình phân rã chuyến đi...
GMM đã phân cụm: 
 - Trễ nội bộ trung bình: 1.4 phút 
 - Nghỉ bến trung bình: 9.8 phút
=> Ngưỡng thời gian cắt chuyến tối ưu: 5.6 phút (337 giây)
1. Đang tiền xử lý và lọc nhiễu dữ liệu...
2. Đang xây dựng đường cong KDE (Làm mượt biểu đồ)...
3. Đang dò tìm Đáy thung lũng (Local Minimum)...
HOÀN TẤT! Ngưỡng cắt chuyến tối ưu là: 18.3 phút (1100 giây)
Đang đánh giá ngưỡng 5.618731859044006 phút...


C:\Users\nguye\AppData\Local\Temp\ipykernel_16412\3503264753.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['prev_is_resting'] = df.groupby('vehicle')['is_resting'].shift(1).fillna(False)
C:\Users\nguye\AppData\Local\Temp\ipykernel_16412\3503264753.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['prev_is_long_idle'] = df.groupby('vehicle')['is_long_idle'].shift(1).fillna(False)


Đang tạo từ điển tra cứu tuyến đường...
Bắt đầu tìm Tuyến lõi
Đang đánh giá ngưỡng 18.337170503837168 phút...


C:\Users\nguye\AppData\Local\Temp\ipykernel_16412\3503264753.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['prev_is_resting'] = df.groupby('vehicle')['is_resting'].shift(1).fillna(False)
C:\Users\nguye\AppData\Local\Temp\ipykernel_16412\3503264753.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['prev_is_long_idle'] = df.groupby('vehicle')['is_long_idle'].shift(1).fillna(False)


Đang tạo từ điển tra cứu tuyến đường...
Bắt đầu tìm Tuyến lõi
Đang đánh giá ngưỡng 15.0 phút...


...

Đang tạo từ điển tra cứu tuyến đường...
Bắt đầu tìm Tuyến lõi
Đang đánh giá ngưỡng 110.0 phút...


C:\Users\nguye\AppData\Local\Temp\ipykernel_16412\3503264753.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['prev_is_resting'] = df.groupby('vehicle')['is_resting'].shift(1).fillna(False)
C:\Users\nguye\AppData\Local\Temp\ipykernel_16412\3503264753.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['prev_is_long_idle'] = df.groupby('vehicle')['is_long_idle'].shift(1).fillna(False)


Đang tạo từ điển tra cứu tuyến đường...
Bắt đầu tìm Tuyến lõi
Đang đánh giá ngưỡng 115.0 phút...


C:\Users\nguye\AppData\Local\Temp\ipykernel_16412\3503264753.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['prev_is_resting'] = df.groupby('vehicle')['is_resting'].shift(1).fillna(False)
C:\Users\nguye\AppData\Local\Temp\ipykernel_16412\3503264753.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['prev_is_long_idle'] = df.groupby('vehicle')['is_long_idle'].shift(1).fillna(False)


Đang tạo từ điển tra cứu tuyến đường...
Bắt đầu tìm Tuyến lõi
BẢNG KẾT QUẢ FINE-TUNING:
 Threshold_Mins  Assigned_Vehicles  Avg_Trips  Avg_Core_Stations
       5.618732                420 134.366667          19.738095
      18.337171                452 125.577434          22.975664
      15.000000                453 125.710817          22.675497
      20.000000                454 125.345815          22.938326
      25.000000                453 125.207506          23.178808
      30.000000                453 124.328918          23.724062
      35.000000                453 123.944812          23.964680
      40.000000                454 123.138767          24.165198
      45.000000                454 122.585903          24.530837
      50.000000                454 122.312775          24.574890
      55.000000                454 122.061674          24.638767
      60.000000                453 122.028698          24.660044
      65.000000                453 121.827815          24.677704
  